### Indoor localization using Wi-Fi

#### Initial Setup

In [24]:
import os
import re
import time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import sklearn as skl
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from matplotlib.ticker import PercentFormatter
from collections import defaultdict

path = "C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project\\Data\\Data Diana\\"

esp_names = ["ESP1", "ESP2", "ESP3", "ESP4"]
num_esps = len(esp_names)

In [25]:
positions = {
    "z00_dia": [f"user00_positionz00_esp0{i}_dia.csv" for i in range(1, 5)],
    "z00_noite": [f"user00_positionz00_esp0{i}_noite.csv" for i in range(1, 5)],
    "a01": [f"user02_positiona01_esp0{i}.csv" for i in range(1, 5)],
    "a02": [f"user02_positiona02_esp0{i}.csv" for i in range(1, 5)],
    "a03": [f"user02_positiona03_esp0{i}.csv" for i in range(1, 5)],
    "a10": [f"user02_positiona10_esp0{i}.csv" for i in range(1, 5)],
    "a11": [f"user02_positiona11_esp0{i}.csv" for i in range(1, 5)],
    "a12": [f"user02_positiona12_esp0{i}.csv" for i in range(1, 5)],
    "b01": [f"user02_positionb01_esp0{i}.csv" for i in range(1, 5)],
    "b02": [f"user02_positionb02_esp0{i}.csv" for i in range(1, 5)],
    "b03": [f"user02_positionb03_esp0{i}.csv" for i in range(1, 5)],
    "b04": [f"user02_positionb04_esp0{i}.csv" for i in range(1, 5)],
    "b05": [f"user02_positionb05_esp0{i}.csv" for i in range(1, 5)],
    "b06": [f"user02_positionb06_esp0{i}.csv" for i in range(1, 5)],
    "b07": [f"user02_positionb07_esp0{i}.csv" for i in range(1, 5)],
    "b08": [f"user02_positionb08_esp0{i}.csv" for i in range(1, 5)],
    "b09": [f"user02_positionb09_esp0{i}.csv" for i in range(1, 5)],
    "b10": [f"user02_positionb10_esp0{i}.csv" for i in range(1, 5)],
    "b11": [f"user02_positionb11_esp0{i}.csv" for i in range(1, 5)],
    "b12": [f"user02_positionb12_esp0{i}.csv" for i in range(1, 5)],
    "c02": [f"user02_positionc02_esp0{i}.csv" for i in range(1, 5)],
    "c03": [f"user02_positionc03_esp0{i}.csv" for i in range(1, 5)],
    "c04": [f"user02_positionc04_esp0{i}.csv" for i in range(1, 5)],
    "c05": [f"user02_positionc05_esp0{i}.csv" for i in range(1, 5)],
    "c06": [f"user02_positionc06_esp0{i}.csv" for i in range(1, 5)],
    "c07": [f"user02_positionc07_esp0{i}.csv" for i in range(1, 5)],
    "c08": [f"user02_positionc08_esp0{i}.csv" for i in range(1, 5)],
    "c09": [f"user02_positionc09_esp0{i}.csv" for i in range(1, 5)],
    "c10": [f"user02_positionc10_esp0{i}.csv" for i in range(1, 5)],
    "c11": [f"user02_positionc11_esp0{i}.csv" for i in range(1, 5)],
    "c12": [f"user02_positionc12_esp0{i}.csv" for i in range(1, 5)],
    "d02": [f"user02_positiond02_esp0{i}.csv" for i in range(1, 5)],
    "d03": [f"user02_positiond03_esp0{i}.csv" for i in range(1, 5)],
    "d04": [f"user02_positiond04_esp0{i}.csv" for i in range(1, 5)],
    "d05": [f"user02_positiond05_esp0{i}.csv" for i in range(1, 5)],
    "d06": [f"user02_positiond06_esp0{i}.csv" for i in range(1, 5)],
    "d07": [f"user02_positiond07_esp0{i}.csv" for i in range(1, 5)],
    "d08": [f"user02_positiond08_esp0{i}.csv" for i in range(1, 5)],
    "d09": [f"user02_positiond09_esp0{i}.csv" for i in range(1, 5)],
    "d10": [f"user02_positiond10_esp0{i}.csv" for i in range(1, 5)],
    "d11": [f"user02_positiond11_esp0{i}.csv" for i in range(1, 5)],
    "d12": [f"user02_positiond12_esp0{i}.csv" for i in range(1, 5)],
    "e05": [f"user02_positione05_esp0{i}.csv" for i in range(1, 5)],
    "e06": [f"user02_positione06_esp0{i}.csv" for i in range(1, 5)],
    "e07": [f"user02_positione07_esp0{i}.csv" for i in range(1, 5)],
}

#### Read and Process the CSI from the CSV

In [26]:
# função que lê valores RSSI do ficheiro CSV - não utilizada [eliminada]

# função que lê e processa os valores CSI do ficheiro CSV
# # semelhante à função de "Wi-Fi-Sensing-[meu].ipynb"
# # mas o ciclo for é usado ao chamar a função e não na função
def read_and_parse_csi(
    file_path: str, num_samples: int = 120, selected_subcarriers: np.ndarray = None
):
    # para cada ficheiro CSV (posição e esp)
    # lemos 120 amostras da coluna 26
    df = pd.read_csv(file_path, header=None)
    csi_raw: pd.Series = df.iloc[:num_samples, 26]
    valid_csi: list[list[int]] = []

    # para cada array da lista de arrays
    for entry in csi_raw:
        if pd.isna(entry) or "[" not in str(entry):
            continue

        try:
            # extrai todos os valores inteiros da string
            nums = [int(n) for n in re.findall(r"-?\d+", str(entry))]

            # se o array tiver 128 valores, adiciona à lista de CSI válidos
            if len(nums) == 128:
                valid_csi.append(nums)
        except ValueError:
            continue

    # converter para array numpy
    valid_csi = np.array(valid_csi)
    if valid_csi.shape[0] == 0:
        return np.empty((0, 0)), np.empty((0,))

    # 64 complex subcarriers
    # converter para números complexos
    complex_csi = valid_csi[:, ::2] + 1j * valid_csi[:, 1::2]

    # FFT shift puts DC in the center (index 32)
    fft_csi = np.fft.fftshift(complex_csi, axes=1)

    # Extract the 52 active subcarriers (IEEE 802.11n standard): [6:58]
    active = fft_csi[:, 6:58]

    # Remove subcarriers at positions 25, 26, 27 (center)
    active = np.delete(active, [25, 26, 27], axis=1)

    # selecting specific subcarriers
    subcarrier_indices = [i for i in range(2, 48)]
    selected_subcarriers = np.array(subcarrier_indices)
    if selected_subcarriers is not None:
        active = active[:, selected_subcarriers]

    # compute módulo dos valores complexos
    magnitudes = np.abs(active)
    return magnitudes

In [27]:
subcarrier_indices: list[int] = [i for i in range(2, 48)]

csi_data: dict[str, dict[str, np.ndarray]] = {}
num_samples: int = 120

# para cada posição / ficheiro associado
for position, files in positions.items():
    for i, file_name in enumerate(files):
        file_path = path + "\\" + file_name
        magnitudes = read_and_parse_csi(file_path, num_samples, subcarrier_indices)

        key = f"esp{i + 1}_{position}"

        # fica com o "csi", não sei porquê...!
        csi_data[key] = {"csi": magnitudes}

In [28]:
csi_data['esp2_d11']['csi'].shape

(70, 46)

#### Dados para Normalização

In [ ]:
normalization_mean: dict[int, np.ndarray] = {}
normalization_abs_max: dict[int, np.ndarray] = {}

esp_ids: list[int] = [1, 2, 3]

# para cada esp, calculamos a média/máx de uma sala vazia
for esp_id in esp_ids:
    # dados CSI de sala vazia
    empty_csi_list = [
        csi_data[k]["csi"]
        for k in csi_data
        if k.startswith(f"esp{esp_id}_") and k.endswith("_dia")
    ]

    if not empty_csi_list:
        print(f"no empty-room data for ESP{esp_id}, skipping")
        continue

    all_empty = np.concatenate(empty_csi_list, axis=0)

    # é calculada a média/máx ao longo de cada coluna (e não linha a linha) (cima para baixo)
    # ou seja, para cada subportadora
    normalization_mean[esp_id] = np.mean(all_empty, axis=0)
    normalization_abs_max[esp_id] = np.max(np.abs(all_empty), axis=0)

In [30]:
normalization_mean[1].shape

(46,)

#### *Def Features*

In [31]:
# função que suaviza (alterações menos abruptas) o sinal
# usando uma média móvel de tamanho n
def smooth(signal: np.ndarray, n: int) -> np.ndarray:
    # n = 20

    # Make an array containing only zeros and with length = length_of_signal + 2*n
    # inicializar o array com zeros
    extremes_zeros: np.ndarray = np.zeros(len(signal) + 2 * n)

    # no fim teremos algo como [s3, s2, s1] + [s0,...,s9] + [s8, s7, s6]
    for i in range(len(signal)):
        if i < n:
            # mirror the start of signal
            extremes_zeros[i] = signal[n - i]

            # mirror the end of signal
            extremes_zeros[len(signal) + n + i] = signal[len(signal) - i - 2]

        # fill the remaining with the signal itself
        extremes_zeros[i + n] = signal[i]

    # Build the array
    smoothen_signal: np.ndarray = np.zeros(len(signal))

    # Fill the array
    # o i percorre o sinal original (a parte do centro)
    for i in range(n, len(extremes_zeros) - n):
        # Calculate the mean of the neighbours - we have to look
        # at the surrounding neighbours, thus we divide the total by 2 look back and further
        mean_neighbours = np.mean(extremes_zeros[i - n // 2 : i + n // 2])
        smoothen_signal[i - n] = mean_neighbours

    return smoothen_signal


# função que segmenta o sinal em janelas de
# tamanho window_size com sobreposição no_overlap
def segments_signal(signal: np.ndarray, window_size: int = 10, no_overlap: int = 2):
    segments: list[np.ndarray] = []

    # não sei para quê
    t_segment: list[np.ndarray] = []

    # percorrer o sinal com passo no_overlap
    for i in range(0, signal.shape[0], no_overlap):
        if i + window_size < signal.shape[0]:
            segment = signal[i : i + window_size]
            segments.append(segment)  # cut segments
            t_segment.append(np.arange(i, i + window_size))  # cut time

    return segments, t_segment

In [72]:
# função que processa o sinal CSI
# # normalização
# # suavização
# # segmentação
# # extração de características

# aqui, entra cada array de esp/posição, 1 a 1
# cada array é normalizado ([sc1 sc2 sc3 ...] - [sc1_mean sc2_mean sc3_mean ...])
def signal_processing_pipeline(
    csi_data: np.ndarray,
    normalization_mean: np.ndarray,
    normalization_abs_max: np.ndarray,
    count
) -> np.ndarray:

    smoothened_data: list[np.ndarray] = []

    # normalizar o sinal
    csi_data_normalized = csi_data - normalization_mean
    csi_data_normalized = csi_data_normalized / normalization_abs_max

    # Iterate over the subsampled data and smoothen the signals
    # para cada sinal (coluna) (dos dados transpostos)
    # vamos suavizá-lo.......... PARA QUÊ??
    for signal in csi_data_normalized.T:
        smoothened_signal = smooth(signal, 20)  # Subsample by a factor of 5
        smoothened_data.append(smoothened_signal)

    # segmentar o sinal
    segments, t_segments = segments_signal(csi_data_normalized)
    segments = np.array(segments)

    # segment tem shape (num_segments, window_size (num_samples), num_subcarriers)
    # axis = 1 é ao longo do tempo (window_size)
    # que resulta em (num_segments x num_subcarriers)
    # calcular média/std ao longo do tempo
    features = [np.mean(segments, axis=1), np.std(segments, axis=1)]
    features = np.array(features)

    # transpor e remodelar a matriz de características
    # mover o eixo 1 para a posição 0 e eixo 0 para a posição 1
    # o eixo 2 mantém-se
    print("\nfeatures_mean = ", features[0]) if count == 1 else None
    print("\nfeatures_std = ", features[1]) if count == 1 else None
    print("\nfeatures.shape = ", features.shape) if count == 1 else None
    X: np.ndarray = np.transpose(features, (1, 0, 2))
    print("\nX = ", X) if count == 1 else None
    print("\nX = ", X.shape) if count == 1 else None
    # shape[0] é o da 1a posição
    # -1 significa "calcula automaticamente"
    X = X.reshape(X.shape[0], -1)
    print("\nX = ", X) if count == 1 else None
    print("\nX = ", X.shape) if count == 1 else None
    return X


# função que gera as matrizes X e Y para a ESP dada
# executada 3 vezes (uma por ESP)
def generate_X_Y_for_esp(
    esp_id: int, normalization_mean: np.ndarray, normalization_abs_max: np.ndarray
):
    X_total: list[np.ndarray] = []
    Y_total: list[list[int]] = []
    count = 0
    # para cada posição/esp
    for key in csi_data:
        if key.startswith(f"esp{esp_id}_"):
            # informação csi
            csi = csi_data[key]["csi"]
            X = signal_processing_pipeline(
                csi, normalization_mean, normalization_abs_max, count
            )
            X_total.append(X)

            # Extração da posição (ex: a01 → coluna=a, linha=01)
            pos = key.split("_")[1]
            col = ord(pos[0]) - ord("a") + 1
            row = int(pos[1:]) if pos[1:].isdigit() else 0
            Y_total.extend([[col, row]] * len(X))
            count += 1

    return np.concatenate(X_total), np.array(Y_total)

#### Machine Learning

In [73]:
XY: dict[int, tuple[np.ndarray, np.ndarray]] = {
    esp: generate_X_Y_for_esp(esp, normalization_mean[esp], normalization_abs_max[esp])
    for esp in esp_ids
}

# os X's ficam matrizes com 90 colunas
# os Y's ficam matrizes com 2 colunas
# cada esp
X_1, Y_1 = XY[1]
X_2, Y_2 = XY[2]
X_3, Y_3 = XY[3]
# X_4, Y_4 = XY[4]


features_mean =  [[ 0.31426686  0.32350959  0.28912993 ... -0.03447519  0.02781658
   0.08147109]
 [ 0.3060936   0.29447538  0.24597    ... -0.02454158  0.02295431
   0.06917026]
 [ 0.34894751  0.37085728  0.35537166 ... -0.03580713  0.03301474
   0.10263763]
 ...
 [ 0.25534982  0.23836245  0.20473803 ... -0.04094295  0.01090681
   0.06171645]
 [ 0.23441398  0.20092981  0.14860047 ... -0.03076157  0.01122144
   0.05149841]
 [ 0.25936827  0.2411196   0.19959616 ... -0.03422211  0.02243739
   0.06961224]]

features_std =  [[0.13917264 0.23606594 0.30633561 ... 0.03325129 0.05168919 0.09210178]
 [0.12924252 0.22865525 0.30151476 ... 0.03882691 0.05186267 0.0850547 ]
 [0.09760681 0.19539901 0.25778066 ... 0.04214579 0.0456146  0.07192071]
 ...
 [0.1558343  0.23459421 0.28910233 ... 0.05987858 0.07490707 0.1040378 ]
 [0.14832567 0.22403557 0.27309724 ... 0.06222802 0.0704364  0.09764996]
 [0.1610168  0.24224455 0.2936593  ... 0.06300126 0.06968509 0.09562732]]

features.shape =  (2, 45, 46

##### 2 Models - 1 for Columns & 1 for Lines

In [37]:
def normalize_confusion_matrix(cm):
    cm = cm.astype(float)
    row_sums = np.sum(cm, axis=1, keepdims=True)
    return cm / row_sums

In [38]:
# X é soma das 3 ESPs
X = np.concatenate([X_1, X_2, X_3])
Y = np.concatenate([Y_1, Y_2, Y_3])
Y = np.array(Y)

print("Unique Columns:", np.unique(Y[:, 0]))
print("Unique Lines  :", np.unique(Y[:, 1]))

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.3, random_state=200
)

column_model = RandomForestClassifier(random_state=200)
column_model.fit(X_train, y_train[:, 0])

line_model = RandomForestClassifier(random_state=200)
line_model.fit(X_train, y_train[:, 1])

# Predictions
pred_col = column_model.predict(X_test)
pred_line = line_model.predict(X_test)

balanced_acc_column = balanced_accuracy_score(y_test[:, 0], pred_col)
balanced_acc_line = balanced_accuracy_score(y_test[:, 1], pred_line)

print("Balanced Accuracy (Column):", balanced_acc_column * 100, "%")
print("Balanced Accuracy (Line)  :", balanced_acc_line * 100, "%")

Unique Columns: [ 1  2  3  4  5 26]
Unique Lines  : [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
Balanced Accuracy (Column): 83.7932350378752 %
Balanced Accuracy (Line)  : 81.41337978492781 %


In [ ]:
# Compute raw confusion matrices
cm_col = confusion_matrix(y_test[:, 0], pred_col)
cm_line = confusion_matrix(y_test[:, 1], pred_line)

# Normalize confusion matrices
confusion_matrix_column = normalize_confusion_matrix(cm_col)
confusion_matrix_line = normalize_confusion_matrix(cm_line)

confusion_matrix_column = confusion_matrix(y_test[:, 0], pred_col, normalize="true")
confusion_matrix_line = confusion_matrix(y_test[:, 1], pred_line, normalize="true")

plt.imshow(confusion_matrix_column, cmap="Spectral", vmin=0, vmax=1)
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Column")

path_to_save = "C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project\\graphs"
plt.savefig(os.path.join(path_to_save, "confusion_matrix_column.png"), format="png")
plt.show()

plt.imshow(confusion_matrix_line, cmap="Spectral", vmin=0, vmax=1)
plt.colorbar()

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Line")

plt.savefig(os.path.join(path_to_save, "confusion_matrix_line.png"), format="png")
plt.show()

##### 1º MODELO:  “Baseline” Random Forest on Raw Features

In [ ]:
X = np.concatenate([X_1, X_2, X_3])
Y_raw = np.concatenate([Y_1, Y_2, Y_3])

col_lin_to_label = {}
label_to_col_lin = {}
next_label = 0

labels = []
for col, lin in Y_raw:
    key = (col, lin)
    if key not in col_lin_to_label:
        col_lin_to_label[key] = next_label
        label_to_col_lin[next_label] = key
        next_label += 1
    labels.append(col_lin_to_label[key])

Y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.3, random_state=200
)

position_model = RandomForestClassifier(random_state=200)
position_model.fit(X_train, y_train)

# Predict
y_pred = position_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy (Combined Position):", accuracy * 100, "%")

In [ ]:
# Raw confusion matrix
cm_pos = confusion_matrix(y_test, y_pred)

# Manual normalization (row-wise)
confusion_matrix_normalized = []
for row_values in cm_pos:
    confusion_matrix_normalized.append(row_values / np.sum(row_values))
cm_pos_norm = np.array(confusion_matrix_normalized)

labels_names = [
    f"{chr(col + ord('a') - 1)}{str(lin).zfill(2)}"
    for (col, lin) in [label_to_col_lin[i] for i in range(len(label_to_col_lin))]
]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm_pos_norm,
    cmap="Spectral",
    vmin=0,
    vmax=1,
    xticklabels=positions,
    yticklabels=positions,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Normalized Confusion Matrix - Combined Position")

plt.savefig(os.path.join(path_to_save, "confusion_matrix_combined.png"), format="png")
plt.show()


ainda não visto:

In [ ]:
def p95(a):
    return float(np.percentile(a, 95)) if len(a) else float("nan")


def timed_read_and_parse(path):
    t0 = time.perf_counter()
    mags = read_and_parse_csi(path, selected_subcarriers=subcarrier_indices)
    timings["parse_ms"].append((time.perf_counter() - t0) * 1e3)
    return mags


def signal_processing_pipeline_timed(
    csi_data, normalization_mean, normalization_abs_max, use_smoothing=True
):
    t0 = time.perf_counter()
    Xn = (csi_data - normalization_mean) / normalization_abs_max
    timings["pre_ms"].append((time.perf_counter() - t0) * 1e3)

    if use_smoothing:
        t1 = time.perf_counter()
        Xs = moving_average(Xn, n=20)
        timings["smooth_ms"].append((time.perf_counter() - t1) * 1e3)
    else:
        Xs = Xn

    t2 = time.perf_counter()
    segments, _ = segments_signal(Xs, window_size=10, no_overlap=2)
    segments = np.array(segments)
    # Example features: mean & std
    feats = np.stack([segments.mean(axis=1), segments.std(axis=1)], axis=1)
    X = feats.reshape(feats.shape[0], -1)
    timings["feat_ms"].append((time.perf_counter() - t2) * 1e3)
    return X


def batch_predict_timed(model, X):
    t0 = time.perf_counter()
    y = model.predict(X)
    timings["infer_ms"].append((time.perf_counter() - t0) * 1e3)
    return y


def summarize(name):
    arr = np.array(timings[name])
    return dict(
        mean_ms=float(arr.mean()), std_ms=float(arr.std()), p95_ms=p95(arr), n=len(arr)
    )

In [ ]:
timings = defaultdict(list)

# para quê?
confusion_matrix_normalized[14]

recall = np.diag(cm_pos) / cm_pos.sum(axis=1)

filtered = {
    lbl: (c, r) for lbl, (c, r) in label_to_col_lin.items() if r >= 1 and c <= 5
}

max_col = 5
max_row = max(r for c, r in filtered.values())

grid = np.full((max_row, max_col), np.nan)

for label, (c, r) in filtered.items():
    grid[r - 1, c - 1] = recall[label]

fig, ax = plt.subplots(figsize=(4, 8))
ax.set_facecolor("#f7f7f7")

norm = mpl.colors.Normalize(vmin=0.50, vmax=1.00)
im = ax.imshow(
    grid,
    origin="lower",
    cmap="RdYlGn",
    interpolation="nearest",
    aspect="equal",
    norm=norm,
)

cb = fig.colorbar(
    im, ax=ax, fraction=0.046, pad=0.04, format=PercentFormatter(xmax=1.0)
)
cb.set_label("Accuracy (%)", fontsize=10, weight="medium")
cb.set_ticks([0.50, 0.60, 0.70, 0.80, 0.90, 1.00])
cb.outline.set_linewidth(0.8)
cb.ax.tick_params(width=0.8)

ax.set_xticks(np.arange(-0.5, max_col, 1), minor=True)
ax.set_yticks(np.arange(-0.5, max_row, 1), minor=True)
ax.grid(which="minor", color="grey", alpha=0.3, linewidth=1)
ax.tick_params(which="minor", length=0)

# Add numeric values
for r in range(max_row):
    for c in range(max_col):
        v = grid[r, c]
        if not np.isnan(v):
            ax.text(
                c,
                r,
                f"{v * 100:.1f}%",
                ha="center",
                va="center",
                fontsize=8,
                fontweight="semibold",
                bbox=dict(
                    facecolor="white", alpha=0.4, boxstyle="round,pad=0.1", linewidth=0
                ),
                color="black",
            )

# Add custom text annotations for ESPs and AP
label_font = dict(
    ha="center", va="center", fontsize=8.5, fontweight="bold", color="black"
)

ax.text(ord("C".lower()) - ord("a"), 0, "ESP1", **label_font)  # C01 → row 0, col 2
ax.text(ord("E".lower()) - ord("a"), 9, "ESP2", **label_font)  # E10 → row 9, col 4
ax.text(ord("E".lower()) - ord("a"), 3, "ESP3", **label_font)  # E04 → row 3, col 4
# ax.text(ord('A'.lower()) - ord('a'), 8, 'ESP4', **label_font)  # A09 → row 8, col 0
ax.text(ord("A".lower()) - ord("a"), 7, "AP", **label_font)  # A08 → row 7, col 0

# Axis ticks and labels
ax.set_xticks(np.arange(max_col))
ax.set_xticklabels([chr(ord("a") + j).upper() for j in range(max_col)], fontsize=12)
ax.set_yticks(np.arange(max_row))
ax.set_yticklabels([f"{i + 1:02d}" for i in range(max_row)], fontsize=10)

ax.set_xlabel("Column", fontsize=14)
ax.set_ylabel("Row", fontsize=14)
# ax.set_title('Probability of correct prediction by room position (using 4 ESPs)', fontsize=12, pad=10)

ax.invert_xaxis()
fig.savefig(
    os.path.join(path_to_save, "heatmap_remove_ESP4_.png"),
    format="png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.2,
)
plt.show()

# --- Warm-up (important for fair timing) ---
_ = position_model.predict(X[:64])

# --- Measure on your test split ---
# Here we simulate real-time by replaying file groups through the pipeline.
# If you want per-file timings, loop over files instead.
N = min(50, len(X_test))  # cap for speed
_ = position_model.predict(X_test[:5])  # extra warmup

t_total = []
for i in range(N):
    t0 = time.perf_counter()
    # We already have X_test ready-made; to include all stages, rebuild features from raw csi for a subset.
    # Example: pick any key with enough samples and run end-to-end once per iteration.
    # If you have a mapping from X to file paths, use it here. Otherwise, just time pre+feat+infer:
    Xi = X_test[i : i + 1]
    _ = batch_predict_timed(position_model, Xi)
    t_total.append((time.perf_counter() - t0) * 1e3)

# --- Summary table ---
summary = {k: summarize(k) for k in timings.keys()}
summary["total_loop_ms"] = dict(
    mean_ms=float(np.mean(t_total)),
    std_ms=float(np.std(t_total)),
    p95_ms=p95(t_total),
    n=len(t_total),
)

summary


Best and Worst Positions

In [ ]:
def pos_to_name(pos):
    col, row = int(pos[0]), int(pos[1])
    return f"{chr(col + ord('A') - 1)}{row:02d}"

In [ ]:
recall_per_label = np.diag(cm_pos_norm)
recall_by_pos = {
    label_to_col_lin[i]: recall_per_label[i] for i in range(len(recall_per_label))
}
sorted_positions = sorted(recall_by_pos.items(), key=lambda kv: kv[1])
filtered_positions = [
    (pos, r) for pos, r in sorted_positions if pos[1] > 0 and pos[0] <= 5
]

worst_positions = [pos for pos, _ in filtered_positions[:3]]
best_positions = [pos for pos, _ in filtered_positions[-3:]]

worst_names = [pos_to_name(p) for p in worst_positions]
best_names = [pos_to_name(p) for p in best_positions]

print("Worst positions:", worst_names)
print("Best  positions:", best_names)

In [ ]:
def plot_confusion_row_ax(pos, cm_norm, label_map, ax, max_cols=5, cmap="Spectral"):
    lbl = next(i for i, (c, r) in label_map.items() if (c, r) == pos)
    row = cm_norm[lbl]
    max_row = max(r for c, r in label_map.values() if c <= max_cols)
    grid = np.full((max_row, max_cols), np.nan)
    for i, p in enumerate(row):
        c, r = label_map[i]
        if 1 <= c <= max_cols and 1 <= r <= max_row:
            grid[r - 1, c - 1] = p
    ax.set_facecolor("#f7f7f7")
    im = ax.imshow(
        grid,
        origin="lower",
        cmap=cmap,
        vmin=0,
        vmax=0.6,
        interpolation="nearest",
        aspect="equal",
    )

    for rr in range(grid.shape[0]):
        for cc in range(grid.shape[1]):
            v = grid[rr, cc]
            if not np.isnan(v) and v > 0:
                ax.text(
                    cc,
                    rr,
                    f"{v * 100:.1f}%",
                    ha="center",
                    va="center",
                    fontsize=8,
                    weight="semibold",
                    bbox=dict(facecolor="white", alpha=0.5, pad=0.1, linewidth=0),
                )

    ax.set_xticks(np.arange(max_cols))
    ax.set_xticklabels([chr(65 + j) for j in range(max_cols)], fontsize=10)
    ax.set_yticks(np.arange(grid.shape[0]))
    ax.set_yticklabels([f"{i + 1:02d}" for i in range(grid.shape[0])], fontsize=10)
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    ax.set_title(f"{chr(pos[0] + 64)}{pos[1]:02d}", pad=6)
    ax.invert_xaxis()
    return im

In [ ]:
fig_w, axs_w = plt.subplots(
    1,
    len(worst_positions),
    figsize=(len(worst_positions) * 4, 8),
    constrained_layout=True,
)

for ax, pos in zip(axs_w, worst_positions):
    im = plot_confusion_row_ax(
        pos, cm_pos_norm, label_to_col_lin, ax, max_cols=5, cmap="Reds"
    )

cbar = fig_w.colorbar(im, ax=axs_w, fraction=0.046, pad=0.04)
cbar.set_label("P(pred | true)", weight="medium")
fig_w.suptitle(
    "Three Worst-Performing Positions", fontsize=16, weight="regular", y=1.02
)

fig_w.savefig(os.path.join(path_to_save, "worst_positions.png"), format="png")
plt.show()


In [ ]:
fig_b, axs_b = plt.subplots(
    1,
    len(best_positions),
    figsize=(len(best_positions) * 4, 8),
    constrained_layout=True,
)
for ax, pos in zip(axs_b, best_positions):
    im = plot_confusion_row_ax(
        pos, cm_pos_norm, label_to_col_lin, ax, max_cols=5, cmap="Greens"
    )
cbar = fig_b.colorbar(im, ax=axs_b, fraction=0.046, pad=0.04)
cbar.set_label("P(pred | true)", weight="medium")
fig_b.suptitle("Three Best-Performing Positions", fontsize=16, weight="regular", y=1.02)


fig_b.savefig(os.path.join(path_to_save, "best_positions.png"), format="png")
plt.show()


##### 2º MODELO: Grid-Search over Random Forest Hyper-parameters - 82,80%

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.3, random_state=200
)

n_estimators = [100, 200]
criterion = ["gini", "entropy"]
max_depth = [None, 10, 20]
min_samples_splits = [2, 5]
min_samples_leafs = [1, 2]

balanced_accuracy_dict = {}
for n in n_estimators:
    for c in criterion:
        for m in max_depth:
            for s in min_samples_splits:
                for l in min_samples_leafs:
                    clf = RandomForestClassifier(
                        n_estimators=n,
                        criterion=c,
                        max_depth=m,
                        min_samples_split=s,
                        min_samples_leaf=l,
                        random_state=200,
                    )
                    clf.fit(X_train, y_train)
                    y_pred = clf.predict(X_test)
                    bal_acc = balanced_accuracy_score(y_test, y_pred)
                    balanced_accuracy_dict[(n, c, m, s, l)] = bal_acc

sorted_results = sorted(
    balanced_accuracy_dict.items(), key=lambda x: x[1], reverse=True
)

print("\nTop 10 Configurations by Accuracy:")
for i, ((n, c, m, s, l), score) in enumerate(sorted_results[:10], 1):
    print(
        f"{i}. Params: n_estimators={n}, criterion={c}, max_depth={m}, "
        f"min_samples_split={s}, min_samples_leaf={l} -> Accuracy: {score:.4f}"
    )


##### 3º MODELO: CNN-Autoencoder + Random Forest on Learned Embeddings - Test Col Acc: 0.7370, Test Row Acc: 0.5322

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Reshape,
    Conv1D,
    Dense,
    Activation,
    Flatten,
    Input,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Conv2D,
    MaxPooling2D,
)
from PIL import Image
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
# 1) find unique labels
unique_cols = np.unique(Y_raw[:,0])
unique_rows = np.unique(Y_raw[:,1])

col_to_idx = {c:i for i,c in enumerate(unique_cols)}
row_to_idx = {r:i for i,r in enumerate(unique_rows)}

y_col_idx = np.array([col_to_idx[c] for c in Y_raw[:,0]])
y_row_idx = np.array([row_to_idx[r] for r in Y_raw[:,1]])

num_cols = len(unique_cols)
num_rows = len(unique_rows)


# Normalizar
scaler = sklearn.preprocessing.MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_padded = np.pad(X_scaled, ((0, 0), (0, 96 - X_scaled.shape[1])), mode='constant')
X_cnn = X_padded.reshape((X_padded.shape[0], 96, 1))


y_col = tf.keras.utils.to_categorical(y_col_idx, num_cols)
y_row = tf.keras.utils.to_categorical(y_row_idx, num_rows)

X_tr, X_temp, ycol_tr, ycol_temp, yrow_tr, yrow_temp =skl.model_selection.train_test_split(X_cnn, y_col, y_row, test_size=0.4, random_state=200)
X_val, X_te, ycol_val, ycol_te, yrow_val, yrow_te = skl.model_selection.train_test_split(X_temp, ycol_temp, yrow_temp, test_size=0.5, random_state=200)

input_dim = 96
input_layer = Input(shape=(input_dim, 1))

# Encoder
x = Conv1D(64, 3, activation='relu', padding='same')(input_layer)
x = tf.keras.layers.MaxPooling1D(2, padding='same')(x)
x = Conv1D(32, 3, activation='relu', padding='same')(x)
x = tf.keras.layers.MaxPooling1D(2, padding='same')(x)
x = Conv1D(16, 3, activation='relu', padding='same')(x)
x = tf.keras.layers.MaxPooling1D(2, padding='same')(x)

# Bottleneck + dense layers
x = Flatten()(x)
x = Dense(2048, activation='relu')(x)
shared = Dense(512, activation='relu')(x)

# Heads
col_out = Dense(num_cols, activation='softmax', name="classifier_col")(shared)
row_out = Dense(num_rows, activation='softmax', name="classifier_row")(shared)

model = Model(inputs=input_layer, outputs=[col_out, row_out])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss={"classifier_col":"categorical_crossentropy",
          "classifier_row":"categorical_crossentropy"},
    metrics={"classifier_col":"accuracy",
             "classifier_row":"accuracy"}
)

model.summary()


# 5) fit with validation_data
history = model.fit(
    X_tr,
    {"classifier_col": ycol_tr, "classifier_row": yrow_tr},
    epochs=50,
    batch_size=64,
    validation_data=(X_val, {"classifier_col": ycol_val, "classifier_row": yrow_val})
)'''

'''# 7) evaluate on test set
evals = model.evaluate(
    X_te,
    {"classifier_col": ycol_te, "classifier_row": yrow_te},
    verbose=1
)
print(f"\nTest Col Acc: {evals[3]:.4f}, Test Row Acc: {evals[4]:.4f}")



# Bottleneck
x = Flatten()(x)

# Classifier
encoded = Dense(2048, activation='linear')(x)
encoded = Dense(512, activation='linear')(encoded)
output_col = Dense(5, activation='softmax', name="classifier_col")(encoded)
output_row = Dense(12, activation='softmax', name="classifier_row")(encoded)

# Build the model
col_model = tf.keras.Model(inputs=input_layer, outputs=output_col)
optimizer= tf_keras.optimizers.Adam(learning_rate=0.0001)
col_model.compile(optimizer=optimizer, loss='categorical_crossentropy')
col_model.summary()

row_model = tf.keras.Model(inputs=input_layer, outputs=output_row)
optimizer= tf_keras.optimizers.Adam(learning_rate=0.0001)
row_model.compile(optimizer=optimizer, loss='categorical_crossentropy')
row_model.summary()


# Decoder
x = Dense((input_dim // 8) * 16, activation='relu')(encoded)
x = Reshape((input_dim // 8, 16))(x)  # Reconstrói para 1D
x = tf.keras.layers.UpSampling1D(2)(x)
x = Conv1D(32, 3, activation='relu', padding='same')(x)
x = tf.keras.layers.UpSampling1D(2)(x)
x = Conv1D(64, 3, activation='relu', padding='same')(x)
x = tf.keras.layers.UpSampling1D(2)(x)
decoded = Conv1D(1, 3, activation='sigmoid', padding='same')(x)

# Autoencoder
autoencoder = tf.keras.Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()


col_model.fit(X_train_auto, np.eye(5)[y_col], epochs=50, batch_size=64)
#one hot vector
row_model.fit(X_train_auto, np.eye(12)[y_row], epochs=50, batch_size=64)
#gravar os modelos depois de treinados
#ver o loss do treino
#meter validação


X_train_encoded = encoder.predict(X_train_auto)
X_test_encoded = encoder.predict(X_test_auto)
encoder = Model(autoencoder.input, autoencoder.get_layer('bottleneck').output)

X_encoded = encoder.predict(X_cnn)

col_lin_to_label = {}
label_to_col_lin = {}
next_label = 0
Y_combined = []
for col, row in Y_raw:
    key = (col, row)
    if key not in col_lin_to_label:
        col_lin_to_label[key] = next_label
        label_to_col_lin[next_label] = key
        next_label += 1
    Y_combined.append(col_lin_to_label[key])
Y_combined = np.array(Y_combined)

X_tr, X_te, y_raw, y_te = sklearn.model_selection.train_test_split(
    X_encoded, Y_combined,
    test_size=0.3, random_state=200
)

clf =  sklearn.ensemble.RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

clf.fit(X_tr, y_raw)

y_pred = clf.predict(X_te)
acc     = sklearn.metrics.accuracy_score(y_te, y_pred)
print(f"Accuracy (combined):          {acc:.4f}")
